In [1]:
import sqlite3
import datetime

# SQLite DB

- 사용자 users
- 몸 상태 conditions
- 대화 내역 conversations
- 이상신호 alerts

## Step 1. DB 연결

In [2]:
conn = sqlite3.connect('dandibom.db')

print('DB 연결 성공')

DB 연결 성공


## Step 2. 테이블 생성

In [3]:
cursor = conn.cursor()

### users 테이블 생성

In [4]:
create_users_table_query = """
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    role VARCHAR(10) NOT NULL CHECK(role IN ('대상자', '보호자', '복지사', '관리자')),
    name VARCHAR(255) NOT NULL,  -- 이름
    code VARCHAR(255) NOT NULL
);
"""

cursor.execute(create_users_table_query)
conn.commit()

print('users 테이블 생성 성공')

users 테이블 생성 성공


### relationships 테이블 생성

In [5]:
create_relationships_table_query = """
CREATE TABLE IF NOT EXISTS relationships (
    rel_id INTEGER PRIMARY KEY AUTOINCREMENT,
    target_id INTEGER,  -- 어르신(대상자)
    guardian_id INTEGER,  -- 보호자
    worker_id INTEGER,  -- 복지사
    FOREIGN KEY (target_id) REFERENCES users(user_id),
    FOREIGN KEY (guardian_id) REFERENCES users(user_id),
    FOREIGN KEY (worker_id) REFERENCES users(user_id)
);
"""

cursor.execute(create_relationships_table_query)
conn.commit()

print('relationships 테이블 생성 성공')

relationships 테이블 생성 성공


### conditions 테이블 생성

In [6]:
create_conditions_table_query = """
CREATE TABLE IF NOT EXISTS conditions (
    cond_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    meal VARCHAR(255) NOT NULL,
    sleep VARCHAR(255) NOT NULL,
    mood VARCHAR(255) NOT NULL,
    pain VARCHAR(255) NOT NULL,
    recorded_at DATETIME NOT NULL,  -- 기록 날짜
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
"""

cursor.execute(create_conditions_table_query)
conn.commit()

print('conditions 테이블 생성 성공')

conditions 테이블 생성 성공


### medications 테이블 생성

In [7]:
create_medications_table_query = """
CREATE TABLE IF NOT EXISTS medications (
    med_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    medicine VARCHAR(255) NOT NULL,
    recorded_at DATETIME NOT NULL,  -- 기록 날짜
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
"""

cursor.execute(create_medications_table_query)
conn.commit()

print('medications 테이블 생성 성공')

medications 테이블 생성 성공


### conversations 테이블 생성

In [8]:
create_conversations_table_query = """
CREATE TABLE IF NOT EXISTS conversations (
    conv_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    role VARCHAR(10) NOT NULL CHECK(role IN ('user', 'assistant')),  -- 어르신, 챗봇
    message TEXT NOT NULL,  -- 대화 내용 (원문 또는 정규화)
    created_at DATETIME NOT NULL,  -- 대화 시각
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
"""

cursor.execute(create_conversations_table_query)
conn.commit()

print('conversations 테이블 생성 성공')

conversations 테이블 생성 성공


### alerts

In [9]:
create_alerts_table_query = """
CREATE TABLE IF NOT EXISTS alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    cause TEXT NOT NULL,  -- 이상신호 근거
    level VARCHAR(10) NOT NULL CHECK(level IN ('낮음', '중간', '높음')),
    status VARCHAR(10) NOT NULL CHECK(status IN ('대기', '확인')),
    created_at DATETIME NOT NULL,  -- 이상신호 발생 시각
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
"""

cursor.execute(create_alerts_table_query)
conn.commit()

print('alerts 테이블 생성 성공')

alerts 테이블 생성 성공


# 샘플 데이터 삽입

## users 테이블

In [10]:
# # users 테이블에 데이터 삽입 1

# insert_user_query = """
# INSERT INTO users (role, name, code) VALUES (?, ?, ?);
# """
# user_data = ('대상자', '박일주', 'user')
# cursor.execute(insert_user_query, user_data)
# conn.commit()

# # 삽입된 사용자의 user_id를 가져옴
# user_id = cursor.lastrowid
# print(f"새로운 사용자 (박일주)의 user_id: {user_id}")

In [11]:
# # users 테이블에 데이터 삽입 2

# # 김순자 데이터 삽입
# insert_user_query = """
# INSERT INTO users (role, name, code) VALUES (?, ?, ?);
# """
# user_data_kim = ('대상자', '김순자', 'user')
# cursor.execute(insert_user_query, user_data_kim)
# conn.commit()
# kim_soonja_id = cursor.lastrowid
# print(f"새로운 사용자 (김순자)의 user_id: {kim_soonja_id}")

# # 이영수 데이터 삽입
# user_data_lee = ('대상자', '이영수', 'user')
# cursor.execute(insert_user_query, user_data_lee)
# conn.commit()
# lee_youngsoo_id = cursor.lastrowid
# print(f"새로운 사용자 (이영수)의 user_id: {lee_youngsoo_id}")

# # 박말순 데이터 삽입
# user_data_park = ('대상자', '박말순', 'user')
# cursor.execute(insert_user_query, user_data_park)
# conn.commit()
# park_malsun_id = cursor.lastrowid
# print(f"새로운 사용자 (박말순)의 user_id: {park_malsun_id}")

In [12]:
users_seed = [
    # (role, name, code)
    ('대상자', '김순자', 'A3F7'),   # ★데모 주인공
    ('보호자', '박지훈', 'A4E8'),
    ('복지사', '이경미', 'A5G9'),
    ('대상자', '박막례', 'B2K9'),
    ('대상자', '이팔복', 'C7M1'),
    ('대상자', '최복순', 'D4P8'),
    ('대상자', '정갑수', 'E1Q3'),
    ('대상자', '한영자', 'F9R6'),
    ('대상자', '오만수', 'G5T2'),
    ('대상자', '신복동', 'H8U7'),
    ('대상자', '강말자', 'J3V4'),
    ('대상자', '윤칠성', 'K6W9'),
    ('대상자', '배순남', 'L2X5'),
    ('대상자', '조덕배', 'M7Y1'),
    ('관리자', '운영자', 'A0Z9'),
]

# users 테이블에 데이터 삽입
insert_users_query = """
INSERT INTO users (role, name, code) VALUES (?, ?, ?);
"""
cursor.executemany(insert_users_query, users_seed)
conn.commit()

print('users 테이블에 샘플 데이터 삽입 성공')

users 테이블에 샘플 데이터 삽입 성공


## relationships 테이블

In [13]:
targets = [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
# Note: user_id 2 (박지훈) is the guardian, user_id 3 (이경미) is the worker.
rel_seed = [(tid, 2 if tid == 1 else None, 3)
            for tid in targets]

cursor.executemany("""INSERT INTO relationships (target_id, guardian_id, worker_id)
                   VALUES (?,?,?)""", rel_seed)
conn.commit()

print('relationships 테이블에 샘플 데이터 삽입 성공')

# > 보호자(박지훈)는 김순자만 연결. 나머지 대상자는 복지사만 담당. → 복지사 대시보드 "담당 대상자 12명"

relationships 테이블에 샘플 데이터 삽입 성공


## conditions 테이블

In [14]:
# # conditions 테이블에 데이터 삽입 1

# insert_condition_query = """
# INSERT INTO conditions (user_id, meal, sleep, mood, pain, medicine, recorded_at) VALUES (?, ?, ?, ?, ?, ?, ?);
# """
# condition_data = (
#     user_id,          # 위에서 얻은 user_id
#     '{"breakfast": "y", "lunch": "n", "dinner": "n"}',              # 식사 (JSON 형식)
#     '보통',      # 수면 (임의의 값)
#     '보통',            # 기분 (임의의 값)
#     '없음',      # 통증
#     '아침',      # 복약 (임의의 값)
#     datetime.date.today().strftime('%Y-%m-%d')  # 현재 날짜 기록
# )
# cursor.execute(insert_condition_query, condition_data)
# conn.commit()

# print(f"conditions 테이블에 user_id {user_id}의 몸 상태 데이터 삽입 성공")

In [15]:
# # conditions 테이블에 데이터 삽입 2

# # 사용자 ID 매핑 (논리적 ID -> 실제 생성된 ID)
# user_id_map = {
#     1: kim_soonja_id, # 김순자
#     2: lee_youngsoo_id, # 이영수
#     3: park_malsun_id # 박말순
# }

# conditions_seed_raw = [
#     # (logical_user_id, meal, sleep, mood, pain, medicine, recorded_at)
#     # --- 김순자(1): 뒤로 갈수록 악화 ---
#     (1, '{"breakfast": "y", "lunch": "y", "dinner": "y"}', "보통",    "보통",   "없음",             "아침",     "2026-07-08"),
#     (1, '{"breakfast": "y", "lunch": "y", "dinner": "n"}', "잠 설침",    "보통",   "약간",         "점심",     "2026-07-09"),
#     (1, '{"breakfast": "n", "lunch": "y", "dinner": "y"}', "보통",    "상쾌함",   "없음",             "저녁",     "2026-07-10"),
#     (1, '{"breakfast": "y", "lunch": "n", "dinner": "y"}', "잠 설침",    "불쾌함",   "약간",         "아침",     "2026-07-11"),
#     (1, '{"breakfast": "n", "lunch": "n", "dinner": "y"}', "잠 설침",    "불쾌함",   "심각",         "점심",   "2026-07-12"),
#     (1, '{"breakfast": "n", "lunch": "n", "dinner": "n"}', "잠 설침",  "불쾌함", "심각",     "저녁",   "2026-07-13"),
#     (1, '{"breakfast": "n", "lunch": "n", "dinner": "n"}', "잠 설침",    "불쾌함", "심각",  "없음","2026-07-14"),  # ★위험
#     # --- 이영수(2): 양호 대비군 ---
#     (2, '{"breakfast": "y", "lunch": "y", "dinner": "y"}', "보통", "상쾌함", "없음",      "아침", "2026-07-12"),
#     (2, '{"breakfast": "y", "lunch": "y", "dinner": "y"}', "보통", "보통", "없음",      "점심", "2026-07-13"),
#     (2, '{"breakfast": "y", "lunch": "y", "dinner": "y"}', "보통", "상쾌함", "약간", "저녁", "2026-07-14"),
#     # --- 박말순(3): 중간·수면 문제 ---
#     (3, '{"breakfast": "y", "lunch": "y", "dinner": "y"}',       "잠 설침",    "보통", "약간", "아침",   "2026-07-12"),
#     (3, '{"breakfast": "n", "lunch": "y", "dinner": "y"}',  "잠 설침",  "불쾌함", "심각",     "점심",   "2026-07-13"),
#     (3, '{"breakfast": "y", "lunch": "y", "dinner": "y"}',       "잠 설침",    "보통", "약간", "저녁", "2026-07-14"),
# ]

# conditions_to_insert = []
# for item in conditions_seed_raw:
#     logical_user_id = item[0]
#     actual_user_id = user_id_map.get(logical_user_id)
#     if actual_user_id is not None:
#         conditions_to_insert.append((actual_user_id,) + item[1:])
#     else:
#         print(f"경고: 논리적 user_id {logical_user_id}에 대한 매핑을 찾을 수 없습니다.")

# insert_condition_query = """
# INSERT INTO conditions (user_id, meal, sleep, mood, pain, medicine, recorded_at) VALUES (?, ?, ?, ?, ?, ?, ?);
# """

# for condition_data in conditions_to_insert:
#     cursor.execute(insert_condition_query, condition_data)
# conn.commit()

# print("conditions 테이블에 추가 샘플 데이터 삽입 성공")

In [16]:
import json

def meal(breakfast, lunch, dinner):
    return json.dumps({
        "breakfast": "y" if breakfast else "n",
        "lunch": "y" if lunch else "n",
        "dinner": "y" if dinner else "n",
    })

def dt(day_offset, hour, minute):
    base_date = datetime.datetime.now()
    # Adjust base_date to be around current date, but with a fixed year for consistent data
    # For demo purposes, setting a fixed date range (e.g., in 2026 as per previous context)
    # If the intent is to use current date, then just use base_date.replace(hour=hour, minute=minute) - datetime.timedelta(days=day_offset)
    # For consistency with commented out cells, let's use a future date like 2026-07-15 as a reference
    reference_date = datetime.datetime(2026, 7, 15)
    return (reference_date - datetime.timedelta(days=day_offset)).replace(hour=hour, minute=minute, second=0, microsecond=0).strftime('%Y-%m-%d %H:%M:%S')

cond_seed = [
    # (user_id, meal(JSON), sleep, mood, pain, recorded_at)
    # ── 김순자(1) : 뒤로 갈수록 악화 → UI "식사 불규칙(최근 2일)"과 일치
    (1, meal(True,  True,  True ), '보통',    '보통',   '없음', dt(4, 9, 10)),
    (1, meal(True,  True,  True ), '보통',    '보통',   '약간', dt(3, 9, 5)),
    (1, meal(True,  False, False), '잠 설침', '불쾌함', '약간', dt(2, 9, 20)),   # ★불규칙 1일차
    (1, meal(False, True,  False), '잠 설침', '불쾌함', '약간', dt(1, 9, 15)),   # ★불규칙 2일차
    # ── 대비군
    (4, meal(True,  True,  True ), '잘잠',    '상쾌함', '없음', dt(0, 8, 40)),   # 박막례 양호
    (5, meal(True,  True,  True ), '보통',    '보통',   '없음', dt(0, 9, 0)),    # 이팔복 양호
]

cursor.executemany("""INSERT INTO conditions (user_id, meal, sleep, mood, pain, recorded_at)
                   VALUES (?,?,?,?,?,?)""", cond_seed)
conn.commit()

print('conditions 테이블에 샘플 데이터 삽입 성공')

conditions 테이블에 샘플 데이터 삽입 성공


## medications 테이블

In [17]:
med_seed = [
    (1, '아침약', dt(2, 8, 30)),
    (1, '점심약', dt(2, 12, 40)),
    (1, '아침약', dt(1, 8, 30)),
    (4, '아침약', dt(0, 8, 0)),
]
cursor.executemany("INSERT INTO medications (user_id, medicine, recorded_at) VALUES (?,?,?)", med_seed)
conn.commit()

print('medications 테이블에 샘플 데이터 삽입 성공')

medications 테이블에 샘플 데이터 삽입 성공


## conversations 테이블

In [18]:
conv_seed = [
    (1, 'assistant', '어르신, 어제는 무릎이 불편하다 하셨는데 오늘은 좀 어떠세요?', dt(1, 10, 0)),
    (1, 'user',      '무릎이 좀 시큰그리하다',                                    dt(1, 10, 1)),
    (1, 'assistant', '많이 불편하시겠어요. 무리하지 마시고 쉬엄쉬엄 하세요.',      dt(1, 10, 2)),
]
cursor.executemany("""INSERT INTO conversations (user_id, role, message, created_at)
                   VALUES (?,?,?,?)""", conv_seed)
conn.commit()

print('conversations 테이블에 샘플 데이터 삽입 성공')

conversations 테이블에 샘플 데이터 삽입 성공


## alerts 테이블

In [19]:
alerts_seed = [
    # (user_id, cause, level, status, created_at)
    # ★김순자 오늘 미확인 1건 = 데모의 결과물  → UI "오늘 10:07"
    (1,  '어지럼·식욕저하 발화 감지',  '중간', '대기', dt(0, 10, 7)),
    # ── 확인완료 13건
    (1,  '수면 불규칙 감지',           '낮음', '확인', dt(5,  9, 20)),
    (1,  '식사 거름 언급',             '낮음', '확인', dt(7,  9, 10)),
    (5,  '수면 불규칙 감지',           '낮음', '확인', dt(3,  9, 40)),
    (4,  '"입맛이 영 없다" 반복 언급',  '중간', '확인', dt(5, 11, 15)),
    (6,  '대화 응답 없음',             '낮음', '확인', dt(6,  9,  0)),
    (7,  '무릎 통증 호소',             '낮음', '확인', dt(8, 10, 30)),
    (8,  '식사 거름 언급',             '낮음', '확인', dt(9,  9, 50)),
    (9,  '수면 불규칙 감지',           '낮음', '확인', dt(10, 10, 10)),
    (10, '기력 저하 언급',             '중간', '확인', dt(11,  9, 30)),
    (11, '통증 호소',                  '낮음', '확인', dt(12, 10,  0)),
    (12, '식욕 저하 언급',             '낮음', '확인', dt(13,  9, 20)),
    (13, '수면 불규칙 감지',           '낮음', '확인', dt(14, 10, 40)),
    (14, '기분 저하 언급',             '낮음', '확인', dt(15,  9, 15)),
]
cursor.executemany("""INSERT INTO alerts (user_id, cause, level, status, created_at)
                   VALUES (?,?,?,?,?)""", alerts_seed)

conn.commit()
print("더미데이터 시딩 완료 — users 15 / relationships 12 / conditions 6 / medications 4 / conversations 3 / alerts 14")
print('alerts 테이블에 샘플 데이터 삽입 성공')

더미데이터 시딩 완료 — users 15 / relationships 12 / conditions 6 / medications 4 / conversations 3 / alerts 14
alerts 테이블에 샘플 데이터 삽입 성공


# 테이블 생성 확인

In [20]:
table_names = ['users', 'relationships', 'conditions', 'conversations', 'medications', 'alerts']

for table_name in table_names:
    try:
        print(f"\n--- {table_name} 테이블 내용 확인 ---")
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()
        if rows:
            # Get column names
            col_names = [description[0] for description in cursor.description]
            print(col_names)
            for row in rows:
                print(row)
        else:
            print("No data in table.")
    except sqlite3.OperationalError as e:
        print(f"Error querying table {table_name}: {e}")


--- users 테이블 내용 확인 ---
['user_id', 'role', 'name', 'code']
(1, '대상자', '김순자', 'A3F7')
(2, '보호자', '박지훈', 'A4E8')
(3, '복지사', '이경미', 'A5G9')
(4, '대상자', '박막례', 'B2K9')
(5, '대상자', '이팔복', 'C7M1')
(6, '대상자', '최복순', 'D4P8')
(7, '대상자', '정갑수', 'E1Q3')
(8, '대상자', '한영자', 'F9R6')
(9, '대상자', '오만수', 'G5T2')
(10, '대상자', '신복동', 'H8U7')
(11, '대상자', '강말자', 'J3V4')
(12, '대상자', '윤칠성', 'K6W9')
(13, '대상자', '배순남', 'L2X5')
(14, '대상자', '조덕배', 'M7Y1')
(15, '관리자', '운영자', 'A0Z9')

--- relationships 테이블 내용 확인 ---
['rel_id', 'target_id', 'guardian_id', 'worker_id']
(1, 1, 2, 3)
(2, 4, None, 3)
(3, 5, None, 3)
(4, 6, None, 3)
(5, 7, None, 3)
(6, 8, None, 3)
(7, 9, None, 3)
(8, 10, None, 3)
(9, 11, None, 3)
(10, 12, None, 3)
(11, 13, None, 3)
(12, 14, None, 3)

--- conditions 테이블 내용 확인 ---
['cond_id', 'user_id', 'meal', 'sleep', 'mood', 'pain', 'recorded_at']
(1, 1, '{"breakfast": "y", "lunch": "y", "dinner": "y"}', '보통', '보통', '없음', '2026-07-11 09:10:00')
(2, 1, '{"breakfast": "y", "lunch": "y", "dinner": "y"}', '보통

# DB 연결 종료

In [21]:
conn.close()